# Zestawienie eksperymentów: Rekonstrukcja Sygnałów na Grafach
Ten notatnik krok po kroku uruchamia wszystkie scenariusze testowe zaimplementowane w projekcie. Został podzielony na dwie części:
1. **Eksperymenty syntetyczne (tabelaryczne)** z modułu `synthetic.py`
2. **Eksperymenty wizualne (wykresy i analizy grafów)** z modułu `testing.py`

**Uwaga:** Upewnij się, że uruchamiasz ten notatnik w głównym katalogu projektu, aby Python widział strukturę plików.


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from src.experiments.synthetic import (
        SyntheticExperimentConfig,
        run_single_source_experiment,
        run_clustering_experiment,
        run_mixture_reconstruction_experiment,
        summarize_synthetic_results
    )
from src.experiments.testing import (
    run_psd_experiment,
    reconstruction_experiment,
    run_mixed_comparison_experiment
)



## 1. Konfiguracja bazowa (`synthetic.py`)
Definiujemy nieco zmniejszoną konfigurację dla szybszego wykonania eksperymentów (np. zmniejszona liczba iteracji `n_runs=3` i rozmiar `n_nodes=200`).


In [5]:
config = SyntheticExperimentConfig(
    topology='knn',
    n_nodes=200,
    n_components=2,
    n_train=300,
    n_test=50,
    observation_probability=0.4,
    n_runs=3,
    seed=42
)
print('Wybrana konfiguracja:')
print(config)


Wybrana konfiguracja:
SyntheticExperimentConfig(topology='knn', profile_name='gaussian', n_nodes=200, n_train=300, n_test=50, n_components=2, observation_probability=0.4, n_runs=3, alpha=10.0, beta=1.0, smooth_beta=0.1, seed=42, knn_neighbors=10, erdos_renyi_probability=0.08, barabasi_albert_m=3, watts_strogatz_neighbors=6, watts_strogatz_rewiring=0.1)


## 2. Eksperyment 1: Pojedyncze Źródło (Single Source)
Weryfikacja jakości estymacji PSD oraz standardowej rekonstrukcji dla danych pochodzących tylko z jednego stacjonarnego rozkładu.


In [6]:
print('Uruchamianie eksperymentu: Pojedyncze Źródło...')
df_single = run_single_source_experiment(config)
summary_single = summarize_synthetic_results(df_single)
display(summary_single)


Uruchamianie eksperymentu: Pojedyncze Źródło...


,experiment,method,metric,mean,std,count
0,single_source_psd,sampled_covariance,psd_mae,0.012430,0.004647,3
1,single_source_psd,sampled_covariance,psd_rmse,0.032667,0.014196,3
2,single_source_reconstruction,single_psd,mae,0.072467,0.012066,3
3,single_source_reconstruction,single_psd,mae_missing,0.081398,0.012996,3
4,single_source_reconstruction,smooth,mae,0.115654,0.006078,3
5,single_source_reconstruction,smooth,mae_missing,0.146487,0.007567,3


## 3. Eksperyment 2: Klastrowanie (Clustering)
Ocena zdolności grupowania (GMM) sygnałów o różnych rozkładach PSD z użyciem transformaty GFT.


In [7]:
print('Uruchamianie eksperymentu: Klastrowanie...')
df_cluster = run_clustering_experiment(config)
summary_cluster = summarize_synthetic_results(df_cluster)
display(summary_cluster)


Uruchamianie eksperymentu: Klastrowanie...


,experiment,method,metric,mean,std,count
0,source_clustering,complete_gft,acc,1.0,0.0,3
1,source_clustering,complete_gft,ari,1.0,0.0,3
2,source_clustering,smooth_gft,acc,1.0,0.0,3
3,source_clustering,smooth_gft,ari,1.0,0.0,3


## 4. Eksperyment 3: Rekonstrukcja Mieszaniny (Mixture Reconstruction)
Kluczowy eksperyment porównujący podejście bazowe (Globalne PSD i Gładkość) z zaproponowanym frameworkiem `clustered_psd`.


In [ ]:
print('Uruchamianie eksperymentu: Rekonstrukcja Mieszaniny...')
df_mix = run_mixture_reconstruction_experiment(config)
summary_mix = summarize_synthetic_results(df_mix)
display(summary_mix)


## 5. Wizualizacja 1: Estymacja PSD przy różnym poziomie obserwacji (`testing.py`)
Prezentuje na wykresach jak zachowuje się estymator kowariancji przy różnych brakach w danych.


In [ ]:
print('Generowanie wykresów: Estymacja PSD...')
_ = run_psd_experiment(N=200, k=20, M=500, p_values=(1.0, 0.5, 0.1))


## 6. Wizualizacja 2: Rekonstrukcja na topologii grafu (Gładka vs PSD)
Kolorowanie węzłów zrekonstruowanymi sygnałami na fizycznym wyglądzie grafu.


In [ ]:
print('Generowanie wykresów: Rekonstrukcja Gładka vs PSD...')
mae_psd, mae_smooth = reconstruction_experiment(N=200, k=20, M_train=500, p=0.4, head=None, seed=42)


## 7. Wizualizacja 3: Mieszanina PSD (Informed vs Global)
Jak globalne PSD zostaje zakłócone przez zmieszanie dwóch różnych źródeł i dlaczego PSD uwzględniające klasy ('Informed') ma przewagę.


In [ ]:
print('Generowanie wykresów: Klasowe vs Globalne PSD dla Mieszaniny...')
df_mixed_vis = run_mixed_comparison_experiment(N=200, k=20, M_train=500, M_test=50, p=0.4, seed=42)
